# 12. 파이썬 기초 - 인코딩과 디코딩

`encoding='cp949'`, `.encode()`, `base64`, `%EC%95%88` …
스크래핑을 하다 보면 계속 마주치지만 **정확히 무슨 일이 일어나는지** 알기 어려운 개념입니다.

이 편의 목표는 딱 하나입니다.

> **"인코딩" 이라는 한 단어가 서로 다른 두 가지 일을 가리킨다는 것을 구분하기**

**다루는 내용**
1. 컴퓨터에는 '글자' 가 없다 — `str` 과 `bytes`
2. encode / decode 기본
3. 규칙이 어긋나면 — 글자 깨짐의 정체
4. 인코딩의 두 층위
5. base64 — 표현 변환
6. URL 인코딩 — 네이버 API 검색어
7. 실전 ① CSV 파일 인코딩
8. 실전 ② requests 의 `.text` vs `.content`

## 1. 컴퓨터에는 '글자' 가 없다

컴퓨터가 저장하고 전송할 수 있는 것은 **숫자(바이트)뿐** 입니다.
`'안'` 이라는 글자 자체를 저장하는 방법은 없습니다.

그래서 **"어떤 글자를 어떤 숫자로 볼지" 약속표** 가 필요합니다.
그 약속표가 **문자 인코딩** (UTF-8, CP949 등) 입니다.

파이썬은 이 둘을 **다른 타입** 으로 명확히 구분합니다.

| 타입 | 정체 | 표기 |
|---|---|---|
| `str` | 사람이 읽는 **글자** | `'안녕'` |
| `bytes` | 컴퓨터가 다루는 **숫자 나열** | `b'\xec\x95\x88...'` |

In [1]:
s = '안녕'

# str : 사람이 읽는 글자
print('타입:', type(s), '/ 길이:', len(s), '글자')

# encode() : 약속표에 따라 숫자로 바꾼다
b = s.encode('utf-8')
print('타입:', type(b), '/ 길이:', len(b), '바이트')

print()
print('bytes 표기 :', b)          # b'...' 로 표시됨
print('실제 숫자  :', list(b))    # 진짜 정체는 그냥 숫자 나열이다

타입: <class 'str'> / 길이: 2 글자
타입: <class 'bytes'> / 길이: 6 바이트

bytes 표기 : b'\xec\x95\x88\xeb\x85\x95'
실제 숫자  : [236, 149, 136, 235, 133, 149]


In [2]:
# 같은 '안녕' 이라도 약속표가 다르면 숫자가 완전히 달라진다
utf8 = '안녕'.encode('utf-8')
cp949 = '안녕'.encode('cp949')

print('UTF-8 :', utf8, '→', list(utf8), f'({len(utf8)}바이트)')
print('CP949 :', cp949, '→', list(cp949), f'({len(cp949)}바이트)')

print()
print('→ UTF-8 은 한글 1글자를 3바이트, CP949 는 2바이트로 표현한다')
print('→ 그래서 같은 글자인데도 파일 크기가 달라진다')

UTF-8 : b'\xec\x95\x88\xeb\x85\x95' → [236, 149, 136, 235, 133, 149] (6바이트)
CP949 : b'\xbe\xc8\xb3\xe7' → [190, 200, 179, 231] (4바이트)

→ UTF-8 은 한글 1글자를 3바이트, CP949 는 2바이트로 표현한다
→ 그래서 같은 글자인데도 파일 크기가 달라진다


In [3]:
# 영문/숫자는 어떤 약속표에서든 대체로 같다 (ASCII 호환)
print("'AB' UTF-8 :", 'AB'.encode('utf-8'), list('AB'.encode('utf-8')))
print("'AB' CP949 :", 'AB'.encode('cp949'), list('AB'.encode('cp949')))

print()
print('→ 영문만 쓸 때는 인코딩 문제를 겪을 일이 거의 없다')
print('→ 한글을 다루는 순간부터 약속표를 신경 써야 한다')

'AB' UTF-8 : b'AB' [65, 66]
'AB' CP949 : b'AB' [65, 66]

→ 영문만 쓸 때는 인코딩 문제를 겪을 일이 거의 없다
→ 한글을 다루는 순간부터 약속표를 신경 써야 한다


## 2. encode / decode 기본

| 용어 | 방향 | 한 문장 |
|---|---|---|
| **인코딩** `encode()` | `str` → `bytes` | 약속표에 따라 **숫자로 바꾸기** |
| **디코딩** `decode()` | `bytes` → `str` | 같은 약속표로 **다시 글자로 읽기** |

> **가장 중요한 원리**
> 인코딩과 디코딩은 **반드시 짝** 이다.
> **넣을 때 쓴 규칙과 꺼낼 때 쓴 규칙이 같아야만** 원본이 나온다.

In [4]:
원본 = '파이썬'

# 왕복: encode → decode
숫자 = 원본.encode('utf-8')     # str  → bytes
복원 = 숫자.decode('utf-8')     # bytes → str

print('원본:', 원본)
print('숫자:', 숫자)
print('복원:', 복원)
print()
print('원본과 같은가?', 원본 == 복원)

# 인자를 생략하면 UTF-8 이 기본값 (파이썬 3 기준)
print("\n.encode() 기본값은 utf-8:", '가'.encode() == '가'.encode('utf-8'))

원본: 파이썬
숫자: b'\xed\x8c\x8c\xec\x9d\xb4\xec\x8d\xac'
복원: 파이썬

원본과 같은가? True

.encode() 기본값은 utf-8: True


## 3. 규칙이 어긋나면 — 글자 깨짐의 정체

실무에서 겪는 **한글 깨짐** 은 파일이 손상된 것이 아닙니다.
**숫자는 멀쩡한데, 읽는 약속표를 잘못 골랐을 뿐** 입니다.

깨지는 방식은 두 가지입니다.

| 유형 | 결과 | 위험도 |
|---|---|---|
| 해석 불가능한 바이트 | `UnicodeDecodeError` **에러 발생** | 낮음 (바로 알아챔) |
| 우연히 해석 가능한 바이트 | **조용히 이상한 글자** 로 변환 | **높음** (모르고 지나감) |

In [5]:
b = '안녕'.encode('utf-8')       # UTF-8 규칙으로 숫자화
print('원본 바이트:', b)

print()
# 유형 A: 같은 규칙 → 정상
print("utf-8 로 디코딩 :", b.decode('utf-8'), ' ← 정상')

print()
# 유형 B: 해석 불가능 → 에러 (그나마 다행인 경우)
try:
    b.decode('cp949')
except UnicodeDecodeError as exp:
    print('cp949 로 디코딩 : UnicodeDecodeError')
    print('   이유:', exp.reason)

원본 바이트: b'\xec\x95\x88\xeb\x85\x95'

utf-8 로 디코딩 : 안녕  ← 정상

cp949 로 디코딩 : UnicodeDecodeError
   이유: illegal multibyte sequence


In [6]:
# errors 옵션: 깨진 글자를 만났을 때의 처리 방법
b2 = '안녕hello'.encode('utf-8')

print("errors='strict'  (기본) → 에러 발생")
try:
    b2.decode('ascii')
except UnicodeDecodeError:
    print('   UnicodeDecodeError')

# 읽을 수 없는 바이트를 버리거나 대체 문자로 바꾼다
print("errors='ignore'  :", repr(b2.decode('ascii', errors='ignore')))
print("errors='replace' :", repr(b2.decode('ascii', errors='replace')))

print()
print('→ ignore/replace 는 데이터가 손실된다. 원인을 모를 때 임시로만 쓸 것')

errors='strict'  (기본) → 에러 발생
   UnicodeDecodeError
errors='ignore'  : 'hello'
errors='replace' : '������hello'

→ ignore/replace 는 데이터가 손실된다. 원인을 모를 때 임시로만 쓸 것


### 3-1. `repr()` — 보이지 않는 문자를 찾아내기

앞의 셀들에서 계속 `repr()` 을 썼습니다. 이유가 있습니다.

| 함수 | 대상 | 목적 |
|---|---|---|
| `print()` / `str()` | **사람** | 보기 좋게 |
| `repr()` | **개발자** | **정확하게** (따옴표·이스케이프 그대로) |

깨진 문자열에는 **화면에 표시되지 않는 제어문자** 가 섞여 있어서,
그냥 `print` 하면 **무슨 값인지 알 수 없습니다.**

```python
print(깨짐)        # ìë                    ← 2글자만 보임
print(repr(깨짐))  # 'ì\x95\x88ë\x85\x95'   ← 실제로는 6글자
```

f-string 안에서는 `{값!r}` 이 `repr(값)` 의 축약 표기입니다.

In [7]:
# =========================================================
# repr() - 눈에 보이지 않는 문자를 찾아내는 도구
# ---------------------------------------------------------
#   print() / str() : '사람이 보기 좋게' 출력
#                     → 공백·개행·BOM 이 화면에서 사라져 구분이 안 된다
#   repr()          : '파이썬이 보는 그대로' 출력
#                     → 따옴표를 붙이고 \n, \t, ﻿ 를 눈에 보이게 만든다
#
#   f-string 안에서는 {값!r} 이 repr(값) 의 축약 표기다.
# =========================================================

# (설명, 실제 값) 쌍. 눈으로는 구분되지 않는 값들을 모았다
samples = [
    ('앞뒤 공백', '  파이썬  '),
    ('개행 포함', '줄1\n줄2'),
    ('탭 포함', '이름\t점수'),
    ('빈 문자열', ''),
    ('BOM 포함', '﻿이름'),
]

for label, value in samples:
    print(f'[{label}]')
    print(f'  print() : {value}')       # 그냥 출력 - 숨은 문자가 안 보인다
    print(f'  repr()  : {value!r}')     # repr 출력 - 숨은 문자가 드러난다
    print(f'  글자 수 : {len(value)}')   # 화면에 보이는 것보다 길 수 있다
    print('-' * 42)

[앞뒤 공백]
  print() :   파이썬  
  repr()  : '  파이썬  '
  글자 수 : 7
------------------------------------------
[개행 포함]
  print() : 줄1
줄2
  repr()  : '줄1\n줄2'
  글자 수 : 5
------------------------------------------
[탭 포함]
  print() : 이름	점수
  repr()  : '이름\t점수'
  글자 수 : 5
------------------------------------------
[빈 문자열]
  print() : 
  repr()  : ''
  글자 수 : 0
------------------------------------------
[BOM 포함]
  print() : ﻿이름
  repr()  : '\ufeff이름'
  글자 수 : 3
------------------------------------------


In [8]:
# =========================================================
# 실전 - '눈으로는 같은데 키가 안 맞는' 상황을 repr 로 진단하기
# =========================================================

# 스크래핑이나 CSV 로 만들어진 딕셔너리라고 가정한다.
#   '이름' 앞에는 BOM(﻿) 이, '나이 ' 뒤에는 공백이 숨어 있다.
data = {'﻿이름': '홍길동', '나이 ': 20}

print('조회 시도:')
print("  data.get('이름') →", data.get('이름'), '← None!')
print("  data.get('나이') →", data.get('나이'), '← None!')

print()
print('화면(print)에는 아무 문제 없어 보인다:')
for k in data:
    print(f'  {k}')

print()
print('repr 로 보면 원인이 드러난다:')
for k in data:
    print(f'  {k!r}  (길이 {len(k)})')

print()
# 컨테이너를 print 하면 내부 요소에는 자동으로 repr 이 적용된다.
# → 딕셔너리/리스트를 통째로 찍어보는 것만으로도 문제를 발견할 수 있다.
print('딕셔너리 통째로 print:', data)

print()
# 원인을 알았으면 정리한다
#   strip()           : 앞뒤 '공백' 을 제거. BOM 은 공백이 아니라서 안 지워진다
#   lstrip('﻿')  : BOM 문자를 직접 지정해 제거
print("'\\ufeff이름'.strip() :", repr('﻿이름'.strip()), '← BOM 은 그대로 남는다')

cleaned = {k.lstrip('﻿').strip(): v for k, v in data.items()}
print()
print('정리 후 키:', [repr(k) for k in cleaned])
print("cleaned['이름'] →", cleaned['이름'])
print("cleaned['나이'] →", cleaned['나이'])

조회 시도:
  data.get('이름') → None ← None!
  data.get('나이') → None ← None!

화면(print)에는 아무 문제 없어 보인다:
  ﻿이름
  나이 

repr 로 보면 원인이 드러난다:
  '\ufeff이름'  (길이 3)
  '나이 '  (길이 3)

딕셔너리 통째로 print: {'\ufeff이름': '홍길동', '나이 ': 20}

'\ufeff이름'.strip() : '\ufeff이름' ← BOM 은 그대로 남는다

정리 후 키: ["'이름'", "'나이'"]
cleaned['이름'] → 홍길동
cleaned['나이'] → 20


## 4. 인코딩의 두 층위

**같은 "인코딩" 이라는 단어가 서로 다른 층위에서 쓰입니다.**

| | **층위 1 — 문자 인코딩** | **층위 2 — 표현 변환** |
|---|---|---|
| 무엇을 바꾸나 | **글자 ↔ 숫자** | **숫자 ↔ 숫자** |
| 타입 | `str` ↔ `bytes` | `bytes` ↔ `bytes` |
| 파이썬 | `.encode()` / `.decode()` | `base64.b64encode()` / `quote()` |
| 예 | UTF-8, CP949, EUC-KR | base64, URL 인코딩, hex |
| 왜 필요한가 | 글자를 저장·전송하려고 | **못 지나가는 통로를 통과시키려고** |

두 층위를 연달아 지나가는 것이 바로 이 코드입니다.

```python
base64.b64encode(비밀번호.encode()).decode()
#                          ①층위1      ③층위1
#                ②층위2
```

In [8]:
import base64

s = '안녕'

step1 = s.encode('utf-8')          # ① 층위1 : str  → bytes
step2 = base64.b64encode(step1)    # ② 층위2 : bytes → bytes
step3 = step2.decode('utf-8')      # ③ 층위1 : bytes → str

print('① .encode()        :', repr(s), '→', step2 and repr(step1))
print('② b64encode()      :', repr(step1), '→', repr(step2))
print('③ .decode()        :', repr(step2), '→', repr(step3))

print()
print('타입 변화 : str → bytes → bytes → str')
print('층위      :  ①1    ②2      ③1')

① .encode()        : '안녕' → b'\xec\x95\x88\xeb\x85\x95'
② b64encode()      : b'\xec\x95\x88\xeb\x85\x95' → b'7JWI64WV'
③ .decode()        : b'7JWI64WV' → '7JWI64WV'

타입 변화 : str → bytes → bytes → str
층위      :  ①1    ②2      ③1


In [ ]:
# 왜 ③ .decode() 가 필요한가?
#   base64 결과는 내용상 알파벳/숫자뿐이지만 '타입' 은 여전히 bytes 다.
#   그대로 출력하면 b'...' 처럼 b 접두사가 붙는다.
print('decode 안 하면:', step2)          # b'7JWI64WV'
print('decode 하면   :', step3)          # 7JWI64WV

print()
# 되돌리기는 정확히 역순  s = '안녕'
back = base64.b64decode(step3).decode('utf-8')
print('복원:', back, '/ 원본과 같은가?', back == s)

decode 안 하면: b'7JWI64WV'
decode 하면   : 7JWI64WV

복원: 안녕 / 원본과 같은가? True


## 5. base64 — 표현 변환은 왜 필요한가

메일 본문, JSON, URL 같은 통로는 **글자로 표현 가능한 것만** 지나갈 수 있습니다.
그런데 이미지 같은 바이너리에는 **글자로 표현할 수 없는 바이트** 가 섞여 있습니다.

그래서 **어디서나 안전한 64개 문자** (`A-Z a-z 0-9 + /`) **만으로 다시 표현** 하는 것이 base64 입니다.

- **장점**: 어떤 통로든 안전하게 통과
- **대가**: 길이가 약 **4/3배** 로 늘어남

In [11]:
data = '안녕'.encode('utf-8')
encoded = base64.b64encode(data)

print('원본 바이트 :', data, f'({len(data)}바이트)')
print('base64      :', encoded, f'({len(encoded)}바이트)')
print(f'길이 비율   : {len(encoded) / len(data):.2f}배')

print()
# base64 결과에는 안전한 ASCII 문자만 들어있다
print('모두 ASCII 인가?', all(chr(c).isascii() for c in encoded))
print('사용된 문자   :', ''.join(sorted(set(encoded.decode()))))

원본 바이트 : b'\xec\x95\x88\xeb\x85\x95' (6바이트)
base64      : b'7JWI64WV' (8바이트)
길이 비율   : 1.33배

모두 ASCII 인가? True
사용된 문자   : 467IJVW


In [12]:
# 실전 예: 이미지를 HTML 에 직접 박아넣기 (data URI)
#   별도 파일 없이 <img> 안에 이미지 자체를 문자열로 넣는 방식
png_bytes = bytes([0x89, 0x50, 0x4E, 0x47, 0x0D, 0x0A, 0x1A, 0x0A])   # PNG 시그니처

# 이 바이트들은 글자로 표현할 수 없다 → 그대로 HTML 에 넣을 수 없다
print('원본 바이트:', png_bytes)

# base64 로 바꾸면 문자열이 되어 HTML 에 넣을 수 있다
b64_str = base64.b64encode(png_bytes).decode()
print('base64     :', b64_str)
print()
print(f'<img src="data:image/png;base64,{b64_str}">')

print()
print('→ 09.넷플릭스 노트북에서 포스터를 이렇게 처리한다')

원본 바이트: b'\x89PNG\r\n\x1a\n'
base64     : iVBORw0KGgo=



→ 09.넷플릭스 노트북에서 포스터를 이렇게 처리한다


## 6. URL 인코딩 — 네이버 API 검색어

URL 에는 **한글, 공백, 특수문자를 그대로 넣을 수 없습니다.**
`?query=파이썬 기초` 처럼 보내면 공백에서 주소가 끊깁니다.

그래서 **안전하지 않은 바이트를 `%XX` (16진수) 로 바꾸는** 것이 URL 인코딩입니다.
이것도 **층위 2** 입니다.

In [13]:
from urllib.parse import quote, unquote, urlencode

keyword = '파이썬 기초'

# quote() : 한글·공백을 %XX 형태로 변환
encoded_kw = quote(keyword)
print('원본     :', keyword)
print('URL 인코딩:', encoded_kw)

print()
# %EC%8C%8C ... 각 %XX 는 UTF-8 바이트 1개를 16진수로 쓴 것이다
print('UTF-8 바이트:', list(keyword.encode('utf-8'))[:6], '...')
print('16진수로    :', [hex(c) for c in keyword.encode('utf-8')[:6]], '...')
print('→ %XX 의 XX 가 바로 이 16진수 값이다')

print()
print('되돌리기 :', unquote(encoded_kw))

원본     : 파이썬 기초
URL 인코딩: %ED%8C%8C%EC%9D%B4%EC%8D%AC%20%EA%B8%B0%EC%B4%88

UTF-8 바이트: [237, 140, 140, 236, 157, 180] ...
16진수로    : ['0xed', '0x8c', '0x8c', '0xec', '0x9d', '0xb4'] ...
→ %XX 의 XX 가 바로 이 16진수 값이다

되돌리기 : 파이썬 기초


In [14]:
# 실전: 네이버 검색 API 요청 URL 만들기
BASE = 'https://openapi.naver.com/v1/search/news.json'

# 방법 1) 직접 조립 - 반드시 quote() 를 거쳐야 한다
url1 = f'{BASE}?query={quote("파이썬")}&display=10'
print('직접 조립:', url1)

# 방법 2) urlencode() - 여러 파라미터를 한 번에 (권장)
params = {'query': '파이썬 기초', 'display': 10, 'sort': 'sim'}
url2 = f'{BASE}?{urlencode(params)}'
print('urlencode:', url2)

print()
# 방법 3) requests 는 params= 로 넘기면 알아서 인코딩해준다 (가장 편함)
print('requests 사용 시: requests.get(BASE, params=params)')
print('  → 내부에서 자동으로 urlencode 를 수행하므로 quote() 를 직접 부를 필요 없다')

직접 조립: https://openapi.naver.com/v1/search/news.json?query=%ED%8C%8C%EC%9D%B4%EC%8D%AC&display=10
urlencode: https://openapi.naver.com/v1/search/news.json?query=%ED%8C%8C%EC%9D%B4%EC%8D%AC+%EA%B8%B0%EC%B4%88&display=10&sort=sim

requests 사용 시: requests.get(BASE, params=params)
  → 내부에서 자동으로 urlencode 를 수행하므로 quote() 를 직접 부를 필요 없다


## 7. 실전 ① CSV 파일 인코딩

**`pd.read_csv()` 의 기본 인코딩은 UTF-8** 입니다.
그런데 국내 공공데이터·엑셀에서 내려받은 CSV 는 대부분 **CP949(=EUC-KR 확장)** 입니다.

파일이 잘못된 게 아니라 **읽는 약속표를 안 알려준 것** 뿐입니다.

In [13]:
import pandas as pd
import io

# CP949 로 저장된 CSV 를 흉내낸다 (실제 파일 대신 메모리에서)
csv_text = '이름,점수\n홍길동,90\n김철수,85\n'
cp949_bytes = csv_text.encode('cp949')

print('파일에 저장된 실제 내용(바이트):', cp949_bytes[:20], '...')

print()
# 잘못된 인코딩으로 읽으면?
try:
    pd.read_csv(io.BytesIO(cp949_bytes))          # 기본값 utf-8
except UnicodeDecodeError as exp:
    print('encoding 지정 안 함 → UnicodeDecodeError')
    print('   이유:', exp.reason)

파일에 저장된 실제 내용(바이트): b'\xc0\xcc\xb8\xa7,\xc1\xa1\xbc\xf6\n\xc8\xab\xb1\xe6\xb5\xbf,90\n' ...

encoding 지정 안 함 → UnicodeDecodeError
   이유: invalid start byte


In [14]:
# 올바른 인코딩을 알려주면 정상적으로 읽힌다
df = pd.read_csv(io.BytesIO(cp949_bytes), encoding='cp949')
print(df)

print()
print('=== 인코딩을 모를 때의 대처 순서 ===')
print('1) encoding="utf-8"      : 요즘 만들어진 파일 대부분')
print('2) encoding="cp949"      : 국내 공공데이터/엑셀 저장 파일')
print('3) encoding="utf-8-sig"  : 엑셀이 만든 UTF-8 (BOM 이 붙어있음)')
print('4) 그래도 안 되면 encoding="latin-1" 로 일단 읽어 내용 확인')

    이름  점수
0  홍길동  90
1  김철수  85

=== 인코딩을 모를 때의 대처 순서 ===
1) encoding="utf-8"      : 요즘 만들어진 파일 대부분
2) encoding="cp949"      : 국내 공공데이터/엑셀 저장 파일
3) encoding="utf-8-sig"  : 엑셀이 만든 UTF-8 (BOM 이 붙어있음)
4) 그래도 안 되면 encoding="latin-1" 로 일단 읽어 내용 확인


In [17]:
# utf-8-sig : 엑셀이 붙이는 BOM(Byte Order Mark) 처리
# BOM 은 파일 맨 앞에 붙는 '눈에 보이지 않는' 3바이트 표식이다.
# "이 파일은 UTF-8 입니다" 라고 알리는 용도지만, 읽는 쪽이 모르면 글자로 딸려 들어온다.
with_bom = '이름,점수\n홍길동,90\n'.encode('utf-8-sig')

print('BOM 포함 바이트 앞부분:', with_bom[:6])
print('→ 앞의 \\xef\\xbb\\xbf 3바이트가 BOM')

print()
# pandas 는 BOM 을 알아서 걸러준다 → read_csv 에서는 문제가 잘 드러나지 않는다
print("pandas encoding='utf-8'     :", list(pd.read_csv(io.BytesIO(with_bom), encoding='utf-8').columns))
print("pandas encoding='utf-8-sig' :", list(pd.read_csv(io.BytesIO(with_bom), encoding='utf-8-sig').columns))
print('→ pandas 는 둘 다 정상이다 (BOM 자동 제거)')

print()
# 문제는 '직접 decode' 할 때 드러난다
print("decode('utf-8')     :", repr(with_bom.decode('utf-8')[:6]), ' ← \\ufeff 가 남아있다')
print("decode('utf-8-sig') :", repr(with_bom.decode('utf-8-sig')[:6]), ' ← 깨끗하다')

BOM 포함 바이트 앞부분: b'\xef\xbb\xbf\xec\x9d\xb4'
→ 앞의 \xef\xbb\xbf 3바이트가 BOM

pandas encoding='utf-8'     : ['이름', '점수']
pandas encoding='utf-8-sig' : ['이름', '점수']
→ pandas 는 둘 다 정상이다 (BOM 자동 제거)

decode('utf-8')     : '\ufeff이름,점수'  ← \ufeff 가 남아있다
decode('utf-8-sig') : '이름,점수\n'  ← 깨끗하다


In [18]:
# BOM 이 실제로 사고를 내는 곳 - csv 모듈과 json 모듈
import csv
import json

s = with_bom.decode('utf-8')          # BOM 을 걸러내지 않고 디코딩

rows = list(csv.reader(io.StringIO(s)))
print('csv.reader 첫 행:', rows[0], ' ← 첫 컬럼명 앞에 BOM 이 붙었다')

d = dict(zip(rows[0], rows[1]))
print("d['이름'] 조회  :", d.get('이름'), ' ← None!')
print('→ 화면에는 똑같이 "이름" 으로 보이는데 실제 키는 "\\ufeff이름" 이라 안 맞는다')

print()
# json 은 아예 에러가 난다 (에러 메시지가 해결책까지 알려준다)
jb = '{"a": 1}'.encode('utf-8-sig')
try:
    json.loads(jb.decode('utf-8'))
except json.JSONDecodeError as exp:
    print('json.loads 실패:', exp.msg)

print('utf-8-sig 로 디코딩하면:', json.loads(jb.decode('utf-8-sig')))

csv.reader 첫 행: ['\ufeff이름', '점수']  ← 첫 컬럼명 앞에 BOM 이 붙었다
d['이름'] 조회  : None  ← None!
→ 화면에는 똑같이 "이름" 으로 보이는데 실제 키는 "\ufeff이름" 이라 안 맞는다

json.loads 실패: Unexpected UTF-8 BOM (decode using utf-8-sig)
utf-8-sig 로 디코딩하면: {'a': 1}


## 8. 실전 ② requests 의 `.text` vs `.content`

웹 응답도 결국 **바이트로 도착** 합니다. 이 둘의 차이를 알면 한글 깨짐을 스스로 고칠 수 있습니다.

| 속성 | 타입 | 설명 |
|---|---|---|
| `res.content` | `bytes` | **서버가 보낸 원본 바이트** (가공 전) |
| `res.text` | `str` | `res.encoding` 규칙으로 **디코딩한 결과** |

즉 `res.text` 는 **`res.content.decode(res.encoding)`** 과 같습니다.
`res.encoding` 을 서버가 잘못 알려주면 `res.text` 가 깨집니다.

In [19]:
import requests

# 네트워크 없이 응답 객체를 직접 만들어 원리만 확인한다
res = requests.models.Response()
res._content = '한글 뉴스 제목'.encode('cp949')   # 서버가 CP949 로 보냈다고 가정
res.status_code = 200

# 서버가 인코딩을 잘못 알려준 상황
res.encoding = 'utf-8'
print("res.encoding = 'utf-8' →", repr(res.text), ' ← 깨짐')

# 올바른 인코딩을 직접 지정하면 정상으로 돌아온다
res.encoding = 'cp949'
print("res.encoding = 'cp949' →", repr(res.text), ' ← 정상')

print()
print('원본 바이트(content):', res.content)
print('→ content 는 항상 그대로다. text 만 encoding 에 따라 달라진다')

res.encoding = 'utf-8' → '�ѱ� ���� ����'  ← 깨짐
res.encoding = 'cp949' → '한글 뉴스 제목'  ← 정상

원본 바이트(content): b'\xc7\xd1\xb1\xdb \xb4\xba\xbd\xba \xc1\xa6\xb8\xf1'
→ content 는 항상 그대로다. text 만 encoding 에 따라 달라진다


In [20]:
# 실전 대처 패턴
print('=== 스크래핑 결과가 깨질 때 ===')
print()
print('# 방법 1) 인코딩을 직접 지정')
print("res = requests.get(url)")
print("res.encoding = 'cp949'      # 또는 'euc-kr'")
print("soup = BeautifulSoup(res.text, 'html.parser')")
print()
print('# 방법 2) 응답 내용을 보고 자동 추정하게 하기')
print("res.encoding = res.apparent_encoding")
print()
print('# 방법 3) 원본 바이트를 넘기고 파서가 알아서 판단하게 하기')
print("soup = BeautifulSoup(res.content, 'html.parser')")

print()
print('→ 방법 3(content 전달)이 가장 무난하다')

=== 스크래핑 결과가 깨질 때 ===

# 방법 1) 인코딩을 직접 지정
res = requests.get(url)
res.encoding = 'cp949'      # 또는 'euc-kr'
soup = BeautifulSoup(res.text, 'html.parser')

# 방법 2) 응답 내용을 보고 자동 추정하게 하기
res.encoding = res.apparent_encoding

# 방법 3) 원본 바이트를 넘기고 파서가 알아서 판단하게 하기
soup = BeautifulSoup(res.content, 'html.parser')

→ 방법 3(content 전달)이 가장 무난하다


## 정리

### 핵심 3가지

1. **인코딩/디코딩은 짝이다.**
   넣을 때 쓴 규칙과 꺼낼 때 쓴 규칙이 같아야만 원본이 나온다.

2. **"인코딩" 에는 두 층위가 있다.**
   - 층위 1 (`str` ↔ `bytes`) : UTF-8, CP949 — `.encode()` / `.decode()`
   - 층위 2 (`bytes` ↔ `bytes`) : base64, URL 인코딩 — 통로를 통과시키려고

3. **인코딩은 암호화가 아니다.**
   규칙이 **공개** 되어 있고 **키가 없다.** 누구나 되돌릴 수 있으므로 값을 숨기는 용도로 쓰면 안 된다.
   (암호화 = "키 없이는 못 푼다" / 인코딩 = "누구나 푼다")

### 문법 요약

| 목적 | 문법 |
|---|---|
| 글자 → 바이트 | `'안녕'.encode('utf-8')` |
| 바이트 → 글자 | `b.decode('utf-8')` |
| 깨져도 진행 | `b.decode('utf-8', errors='replace')` |
| base64 | `base64.b64encode(b)` / `b64decode(s)` |
| URL 인코딩 | `quote('파이썬')` / `urlencode(params)` |
| CSV 읽기 | `pd.read_csv(f, encoding='cp949')` |
| 엑셀 CSV (BOM) | `encoding='utf-8-sig'` |
| 웹 응답 | `res.encoding = 'cp949'` 또는 `BeautifulSoup(res.content, ...)` |

### 증상별 진단표

| 증상 | 원인 | 해결 |
|---|---|---|
| `UnicodeDecodeError` | 읽는 인코딩이 틀림 | `encoding=` 을 cp949 / utf-8-sig 로 |
| 글자가 `ìì` 처럼 깨짐 | 조용히 잘못 디코딩됨 | 올바른 인코딩 지정 |
| 키 이름이 같아 보이는데 안 맞음 | 앞에 BOM(`\ufeff`) | `encoding='utf-8-sig'` 로 디코딩 |
| `Unexpected UTF-8 BOM` (json) | 앞에 BOM | `.decode('utf-8-sig')` 후 파싱 |
| URL 요청이 실패 | 한글/공백 미인코딩 | `quote()` 또는 `params=` |

> **BOM 주의**: `pandas.read_csv` 는 BOM 을 **자동으로 걸러줍니다.**
> 그래서 pandas 만 쓰면 BOM 문제를 못 느끼다가,
> `open()` + `csv` 모듈이나 `json` 모듈로 직접 읽는 순간 처음 마주치게 됩니다.

다음: `08.Selenium사용_GitHub로그인.ipynb` 의 base64 셀을 다시 보면 왜 `.encode()` → `b64encode()` → `.decode()` 3단계인지 이해될 것입니다.